# 00 Baseline Eval

Kaggle-first Milestone 1 baseline evaluation notebook.

This notebook keeps orchestration in the notebook and shared logic in `src/`.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    for candidate in [ROOT, *ROOT.parents]:
        if (candidate / "src").exists():
            ROOT = candidate
            break

if not (ROOT / "src").exists() and Path("/kaggle/input").exists():
    for nested in Path("/kaggle/input").rglob("src"):
        if nested.is_dir():
            ROOT = nested.parent
            break

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

WORKDIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else ROOT
IS_KAGGLE = Path("/kaggle").exists()
ROOT, WORKDIR, IS_KAGGLE


In [ ]:
from src import load_config
from src.data import load_eval_examples, resolve_dataset_path
from src.eval import build_predictor, evaluate_examples
from src.eval.reporting import export_handoff_bundle
from src.eval.splits import split_examples
from src.prompts import get_prompt_variants
from src.solvers import ConservativeRouter

config = load_config(ROOT / "configs/experiments/baseline_eval.yaml")
if IS_KAGGLE and config["predictor"].get("type") == "heuristic":
    config = {
        **config,
        "predictor": {
            **config["predictor"],
            "type": "transformers_kaggle",
        },
    }
    print("Running on Kaggle: overriding predictor.type to transformers_kaggle")

train_path = resolve_dataset_path(
    config["data"].get("train_path"),
    fallback_filename="train.csv",
    auto_discover=config["data"].get("auto_discover", False),
    base_dir=ROOT,
)
print("Using train file:", train_path)

all_examples = load_eval_examples(train_path)
train_examples, validation_examples = split_examples(
    all_examples,
    validation_ratio=config["data"]["validation_ratio"],
    seed=config["data"]["seed"],
)
selected_examples = validation_examples or all_examples
eval_limit = config["data"].get("eval_limit")
if eval_limit is not None:
    selected_examples = selected_examples[: eval_limit]

len(train_examples), len(selected_examples)


In [ ]:
selected_prompt_names = set(config["prompts"])
prompt_variants = [
    variant
    for variant in get_prompt_variants()
    if variant.name in selected_prompt_names
]
router = ConservativeRouter(
    confidence_threshold=config["routing"]["confidence_threshold"],
    enabled_families=tuple(config["routing"].get("enabled_families", [])) or None,
)
predictor = build_predictor(config)

result = evaluate_examples(
    selected_examples,
    prompt_variants=prompt_variants,
    predictor=predictor,
    router=router if config["routing"]["enabled"] else None,
    run_name=config["run_name"],
    output_dir=WORKDIR / config["reporting"]["output_dir"],
    failure_sample_size=config["reporting"].get("failure_sample_size", 5),
)
handoff_bundle = export_handoff_bundle(
    result,
    WORKDIR / config["kaggle"]["output_dir"],
    selected_variant=config["selected_prompt_variant"],
)
result.metrics


In [ ]:
summary_path = result.artifact_paths["summary_md"]
print(summary_path.read_text(encoding="utf-8"))
print(f"Handoff bundle: {handoff_bundle}")
